# Run all notebooks in the project

In [ ]:
# -*- coding: utf-8 -*-
"""Comparaison de la Phase 3 entre DEUX sujets (ex. sain vs pathologique).
Pour chaque sujet : on choisit un fichier + un numero de manoeuvre, puis on
ajuste a la souris les bornes de la Phase 3 (traits glissables, proposition =
coude par l'angle alpha). Les deux manoeuvres sont ensuite superposees sur un
meme graphe, memes echelles, pour comparer les pentes.

Lancer comme un SCRIPT (python capno_comparaison.py), pas dans Jupyter."""

import matplotlib
matplotlib.use("TkAgg")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from scipy.signal import savgol_filter, find_peaks
from scipy.stats import linregress

# =========================================================================== #
#  A ADAPTER : les deux sujets a comparer                                     #
# =========================================================================== #
FICHIER_A   = "/Users/camille/Desktop/EIAP/AnalyseCO/AnalyseCO/data/Conc_MD01_P55_20ms.csv"
LABEL_A = "Healthy subject"
MANOEUVRE_A = 1                        # numero de la manoeuvre a utiliser pour A

FICHIER_B   = "/Users/camille/Desktop/EIAP/AnalyseCO/AnalyseCO/data/Conc_MD01_P36_20ms.csv"
LABEL_B = "Patient"
MANOEUVRE_B = 2                        # numero de la manoeuvre a utiliser pour B

DOSSIER_SORTIE = os.path.join(os.path.expanduser("~"), "Desktop")

TIME_COL = "time_clock_medisoft_complet (s)"
CO2_COL  = "CO2_medisoft_complet (%)"
SMOOTH_S, HEIGHT_FRAC, PROM_FRAC = 0.50, 0.45, 0.30
WIDTH_MIN_S, DIST_MIN_S, DUREE_MIN_S, APRES_S = 2.0, 4.0, 4.0, 10.0


def fitline(x, y):
    s, b, r, _, _ = linregress(x, y)
    return s, b, r ** 2

def charge(f):
    df = pd.read_csv(f, sep=";")
    t = pd.to_numeric(df[TIME_COL], errors="coerce").to_numpy()
    c = pd.to_numeric(df[CO2_COL], errors="coerce").to_numpy()
    ok = ~np.isnan(t) & ~np.isnan(c); t, c = t[ok], c[ok]
    dt = float(np.median(np.diff(t)[np.diff(t) > 0]))
    tg = np.arange(t[0], t[-1], dt); cg = np.interp(tg, t, c)
    w = max(7, int(SMOOTH_S / dt) | 1); w = min(w, len(cg) - (1 - len(cg) % 2))
    return tg, savgol_filter(cg, w, 3), dt

def detecte(tg, cs, dt):
    amp, base = cs.max() - cs.min(), cs.min()
    peaks, _ = find_peaks(cs, height=base + HEIGHT_FRAC * amp, prominence=PROM_FRAC * amp,
                          width=int(WIDTH_MIN_S / dt), distance=int(DIST_MIN_S / dt))
    segs = []
    for p in peaks:
        margin = max(0.15, 0.10 * (cs[p] - base)); o, rm, foot = p, cs[p], p
        while o > 0:
            o -= 1
            if cs[o] < rm: rm, foot = cs[o], o
            elif cs[o] > rm + margin: break
        if (tg[p] - tg[foot]) >= DUREE_MIN_S:
            segs.append((foot, p))
    return segs

def knee_angle(tt, cc):
    xn = (tt - tt[0]) / (tt[-1] - tt[0]); yn = (cc - cc.min()) / (cc.max() - cc.min() + 1e-9)
    nn = len(tt); lo, hi = max(int(0.10 * nn), 4), int(0.90 * nn); best = None
    for k in range(lo, hi):
        m1, _, _ = fitline(xn[:k + 1], yn[:k + 1]); m2, _, _ = fitline(xn[k:], yn[k:])
        if m1 <= m2: continue
        u1 = np.array([1, m1]) / np.hypot(1, m1); u2 = np.array([1, m2]) / np.hypot(1, m2)
        e = np.degrees(np.arccos(np.clip(u1 @ u2, -1, 1)))
        if best is None or e > best[0]: best = (e, k)
    return best[1] if best else int(0.6 * len(tt))


# =========================================================================== #
#  Fenetre interactive pour choisir les bornes de Phase 3 d'UN sujet          #
# =========================================================================== #
def choisir(f, label, man_index):
    tg, cs, dt = charge(f); segs = detecte(tg, cs, dt)
    if not segs:
        raise RuntimeError(f"Aucune manoeuvre detectee dans {f}")
    a, b = segs[min(max(man_index, 1), len(segs)) - 1]
    k = knee_angle(tg[a:b + 1], cs[a:b + 1])
    start0, end0 = float(tg[a + k]), float(tg[b])
    bext = min(b + int(APRES_S / dt), len(tg) - 1)
    tt, cc = tg[a:bext + 1], cs[a:bext + 1]
    st = {"start": start0, "end": end0, "drag": None}

    fig, ax = plt.subplots(figsize=(9, 5.2)); plt.subplots_adjust(bottom=0.20)
    ax.plot(tt, cc, lw=0.9, color="#4a90d9", label="CO2")
    l_start = ax.axvline(start0, color="green", lw=1.6, label="debut P3")
    l_end = ax.axvline(end0, color="purple", lw=1.6, label="fin P3")
    (regline,) = ax.plot([], [], "-", color="red", lw=2.2)
    ax.set_xlabel("temps (s)"); ax.set_ylabel("CO2 (%)"); ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)

    def recompute():
        lo, hi = sorted((st["start"], st["end"])); m = (tt >= lo) & (tt <= hi)
        if m.sum() >= 3:
            s, bb, r2 = fitline(tt[m], cc[m]); xs = tt[m]
            regline.set_data(xs, s * xs + bb)
            ax.set_title(f"{label} — manoeuvre {man_index} — P3 = {s:.3f} %/s (r2={r2:.3f})  [regle puis Valider]")
        fig.canvas.draw_idle()

    def nearest(x): return "start" if abs(x - st["start"]) <= abs(x - st["end"]) else "end"
    def on_press(e):
        if e.inaxes != ax or e.xdata is None: return
        st["drag"] = nearest(e.xdata); x = min(max(e.xdata, tt[0]), tt[-1]); st[st["drag"]] = x
        (l_start if st["drag"] == "start" else l_end).set_xdata([x, x]); recompute()
    def on_motion(e):
        if st["drag"] is None or e.inaxes != ax or e.xdata is None: return
        x = min(max(e.xdata, tt[0]), tt[-1]); st[st["drag"]] = x
        (l_start if st["drag"] == "start" else l_end).set_xdata([x, x]); recompute()
    def on_release(e): st["drag"] = None
    fig.canvas.mpl_connect("button_press_event", on_press)
    fig.canvas.mpl_connect("motion_notify_event", on_motion)
    fig.canvas.mpl_connect("button_release_event", on_release)
    bval = Button(plt.axes([0.80, 0.05, 0.15, 0.075]), "Valider ->")
    bval.on_clicked(lambda ev: plt.close(fig))
    def do_reset(ev):
        st["start"], st["end"] = start0, end0
        l_start.set_xdata([start0, start0]); l_end.set_xdata([end0, end0]); recompute()
    Button(plt.axes([0.62, 0.05, 0.16, 0.075]), "Reset auto").on_clicked(do_reset)
    recompute(); plt.show()

    lo, hi = sorted((st["start"], st["end"])); m = (tt >= lo) & (tt <= hi)
    s, bb, r2 = fitline(tt[m], cc[m])
    return dict(label=label, tt=tt, cc=cc, lo=lo, hi=hi, s=s, r2=r2)


# =========================================================================== #
#  Selection des deux sujets puis superposition                               #
# =========================================================================== #
A = choisir(FICHIER_A, LABEL_A, MANOEUVRE_A)
B = choisir(FICHIER_B, LABEL_B, MANOEUVRE_B)

couleurs = {LABEL_A: "#1b7837", LABEL_B: "#b2182b"}
fig, ax1 = plt.subplots(figsize=(9, 6))
for S in (A, B):
    col = couleurs[S["label"]]
    m = (S["tt"] >= S["lo"]) & (S["tt"] <= S["hi"]); xs = S["tt"][m] - S["lo"]
    s, bb, r2 = fitline(xs, S["cc"][m])
    ax1.plot(S["tt"] - S["lo"], S["cc"], lw=1, color=col, alpha=0.5,
             label=f"{S['label']}: slope {s:.3f} %/s (r2={r2:.2f})")
    ax1.plot(xs, s * xs + bb, "-", color=col, lw=2.6)
ax1.axvline(0, color="gray", ls=":")
ax1.set_title("Expiratory maneuvers aligned at Phase 3 onset (absolute CO2)")
ax1.set_xlabel("time from Phase 3 onset (s)"); ax1.set_ylabel("CO2 (%)")
ax1.grid(alpha=0.3); ax1.legend()
fig.tight_layout()
fig.savefig(os.path.join(DOSSIER_SORTIE, "comparaison_phase3.png"), dpi=120)
print(f"{LABEL_A}: Phase 3 slope = {A['s']:.4f} %/s (r2={A['r2']:.3f})")
print(f"{LABEL_B}: Phase 3 slope = {B['s']:.4f} %/s (r2={B['r2']:.3f})")
print("Figure saved: comparaison_phase3.png (Desktop)")
plt.show()

Healthy subject: Phase 3 slope = 0.0250 %/s (r2=0.883)
Patient: Phase 3 slope = 0.0889 %/s (r2=0.990)
Figure saved: comparaison_phase3.png (Desktop)
